# Fabric MLV file ingestion: self-contained run
Attach to the schema-enabled lakehouse, run top to bottom. Source: `table_a_files_notebook.txt`.

In [ ]:
ctx = notebookutils.runtime.context
WORKSPACE = ctx["currentWorkspaceName"]      # or set by hand
LAKEHOUSE = ctx["defaultLakehouseName"]      # the attached default lakehouse
assert " " not in WORKSPACE + LAKEHOUSE, "Names with spaces are not accepted in the path; use names without spaces"
FOLDER = (f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
          f"{LAKEHOUSE}.Lakehouse/Files/mlv_demo/table_a_files/")

FILE_001 = """id,segment,attribute_a,value
1,A,a1,10.00
2,A,a2,20.00
3,B,a1,15.00
4,B,a3,5.00
5,C,a2,30.00
,A,a1,99.00
6,C,a3,-4.00
"""
notebookutils.fs.put(FOLDER + "file_001.csv", FILE_001, True)

# The bronze %%sql cell reads this value through Spark variable substitution.
spark.conf.set("demo.table_a_files_path", FOLDER)
print(FOLDER)
print(notebookutils.fs.ls(FOLDER))

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [ ]:
%%sql
-- If the parser reports '${demo.table_a_files_path}' literally, replace it with the
-- path printed by CELL 1 (keep the trailing slash).
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS bronze.table_a_files
USING OneLake_Files
OPTIONS (
    'format' = 'csv',
    'path'   = '${demo.table_a_files_path}',
    'header' = 'true'
)
TBLPROPERTIES (
    'schema_mode'  = 'DYNAMIC',
    'refresh_mode' = 'APPEND_ONLY'
);

In [ ]:
%%sql
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS silver.table_a_files_clean
(
    CONSTRAINT id_not_null         CHECK (id IS NOT NULL)              ON MISMATCH DROP,
    CONSTRAINT non_negative_value  CHECK (value >= 0)                  ON MISMATCH DROP,
    CONSTRAINT known_segment       CHECK (segment IN ('A', 'B', 'C'))  ON MISMATCH DROP
)
TBLPROPERTIES (delta.enableChangeDataFeed=true)
AS SELECT
    CAST(id AS INT)               AS id,
    segment,
    attribute_a,
    CAST(value AS DECIMAL(10,2))  AS value,
    __filepath__                  AS source_file
FROM bronze.table_a_files;

In [ ]:
%%sql
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS gold.value_by_segment_files
AS SELECT
    segment,
    COUNT(*)    AS row_count,
    SUM(value)  AS total_value
FROM silver.table_a_files_clean
GROUP BY segment;

In [ ]:
%%sql
-- Expected: A 2 / 30.00, B 2 / 20.00, C 1 / 30.00 (bronze 7 rows, silver 5, 2 dropped).
SELECT * FROM gold.value_by_segment_files ORDER BY segment;

In [ ]:
FILE_002 = """id,segment,attribute_a,value
7,A,a3,12.00
8,B,a2,8.00
9,C,a1,25.00
10,X,a1,50.00
"""
notebookutils.fs.put(FOLDER + "file_002.csv", FILE_002, True)
print(notebookutils.fs.ls(FOLDER))

In [ ]:
%%sql
-- Preferred in a demo: Lakehouse > Materialized lake views > Manage > Run (refresh with
-- dependent lineage), which also shows files processed and rows added. This cell is the
-- notebook equivalent. A notebook refresh does not cascade, so each view is refreshed
-- in order. FULL on bronze re-reads the folder as a snapshot, so no duplicate rows.
REFRESH MATERIALIZED LAKE VIEW bronze.table_a_files FULL;
REFRESH MATERIALIZED LAKE VIEW silver.table_a_files_clean FULL;
REFRESH MATERIALIZED LAKE VIEW gold.value_by_segment_files FULL;

In [ ]:
%%sql
-- Expected: file_001.csv 7 / 5 / 2 dropped, file_002.csv 4 / 3 / 1 dropped.
SELECT element_at(split(b.__filepath__, '/'), -1)  AS file,
       b.bronze_rows,
       COALESCE(s.silver_rows, 0)                  AS silver_rows,
       b.bronze_rows - COALESCE(s.silver_rows, 0)  AS dropped_rows
FROM (SELECT __filepath__, COUNT(*) AS bronze_rows FROM bronze.table_a_files GROUP BY __filepath__) b
LEFT JOIN (SELECT source_file, COUNT(*) AS silver_rows FROM silver.table_a_files_clean GROUP BY source_file) s
  ON s.source_file = b.__filepath__
ORDER BY file;

Lakehouse > Materialized lake views > Manage > Schedule: add a schedule (5 minute
minimum) so new files are picked up without this notebook. Delete it after the demo.
Cleanup (only when done), downstream first:
  DROP MATERIALIZED LAKE VIEW gold.value_by_segment_files;
  DROP MATERIALIZED LAKE VIEW silver.table_a_files_clean;
  DROP MATERIALIZED LAKE VIEW bronze.table_a_files;
  then delete Files/mlv_demo/table_a_files/ in the Lakehouse explorer.

In [ ]:
%%sql
-- Expected:
--   segment | row_count | total_value
--   A       | 3         | 42.00
--   B       | 3         | 28.00
--   C       | 2         | 55.00
SELECT segment, row_count, total_value
FROM gold.value_by_segment_files
ORDER BY segment;